# FitMyResume Phase 9 Notebook Demo

This notebook demonstrates the end-to-end FitMyResume workflow for the CS455 project: provide a resume and job description, then display a fit score, explanation, and resume improvement suggestions.

The default demo path uses the saved fine-tuned validation sample in `results/finetuned_qwen_validation_transformers_sample50_outputs.jsonl`, so it can run without a GPU. Optional Transformers/PEFT live inference notes are included near the end for GPU-backed demos.

## Demo Backend

- **Default:** saved fine-tuned model output from the Transformers/PEFT validation sample.
- **Optional live path:** run `src/run_finetuned_transformers_inference.py` with the local LoRA adapter.
- **Fallback reason:** local notebooks often do not have enough GPU memory for Qwen2.5-7B plus LoRA inference, so the saved-output path is the reliable presentation demo.

## Colab / VS Code Colab Setup

Run this cell first when using Google Colab directly or VS Code connected to a Google Colab runtime. It mounts Google Drive and changes into the project folder.

In [1]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content
%cd /content/drive/MyDrive/fit-my-resume

Mounted at /content/drive
/content
/content/drive/MyDrive/fit-my-resume


In [2]:
from __future__ import annotations

import json
from pathlib import Path
from textwrap import shorten
from typing import Any

MAX_INPUT_CHARS = 12000
RESULTS_PATH = Path("results/finetuned_qwen_validation_transformers_sample50_outputs.jsonl")
INSTRUCTIONS_PATH = Path("data/instruction_tuning/instruction_tuning_validation.jsonl")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root. Run this notebook from the project checkout.")


PROJECT_ROOT = find_project_root()
RESULTS_FILE = (PROJECT_ROOT / RESULTS_PATH).resolve()
INSTRUCTIONS_FILE = (PROJECT_ROOT / INSTRUCTIONS_PATH).resolve()
print(f"Project root: {PROJECT_ROOT}")
print(f"Demo artifact: {RESULTS_FILE}")
print(f"Instruction inputs: {INSTRUCTIONS_FILE}")

Project root: /content/drive/MyDrive/fit-my-resume
Demo artifact: /content/drive/MyDrive/fit-my-resume/results/finetuned_qwen_validation_transformers_sample50_outputs.jsonl
Instruction inputs: /content/drive/MyDrive/fit-my-resume/data/instruction_tuning/instruction_tuning_validation.jsonl


In [3]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row, dict):
                raise ValueError(f"{path} line {line_number} is not a JSON object")
            rows.append(row)
    return rows


def split_instruction_input(input_text: str) -> tuple[str, str]:
    resume_marker = "RESUME:\n"
    job_marker = "\n\nJOB_DESCRIPTION:\n"
    if resume_marker not in input_text or job_marker not in input_text:
        return input_text.strip(), ""
    resume_text = input_text.split(resume_marker, 1)[1].split(job_marker, 1)[0]
    job_text = input_text.split(job_marker, 1)[1]
    return resume_text.strip(), job_text.strip()


def validate_input_lengths(resume_text: str, job_description: str, max_chars: int = MAX_INPUT_CHARS) -> None:
    resume_chars = len(resume_text)
    job_chars = len(job_description)
    if resume_chars > max_chars:
        raise ValueError(f"Resume is {resume_chars:,} characters; limit is {max_chars:,}.")
    if job_chars > max_chars:
        raise ValueError(f"Job description is {job_chars:,} characters; limit is {max_chars:,}.")


def get_parsed_output(row: dict[str, Any]) -> dict[str, Any]:
    parsed = row.get("parsed_output")
    if isinstance(parsed, dict):
        return parsed

    raw_response = row.get("raw_response", "")
    if isinstance(raw_response, str) and raw_response.strip():
        return json.loads(raw_response)

    raise ValueError("Row does not contain a parsed output or JSON raw response.")


def find_demo_row(rows: list[dict[str, Any]]) -> dict[str, Any]:
    parseable_rows = [row for row in rows if row.get("parse_success") and isinstance(row.get("parsed_output"), dict)]
    if not parseable_rows:
        raise ValueError("No parseable fine-tuned outputs found in the demo artifact.")
    return max(parseable_rows, key=lambda row: int(row.get("parsed_output", {}).get("score", -1)))


def instruction_rows_by_pair_id(rows: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    by_pair_id: dict[str, dict[str, Any]] = {}
    for row in rows:
        metadata = row.get("metadata", {})
        if not isinstance(metadata, dict):
            continue
        pair_id = metadata.get("pair_id")
        if isinstance(pair_id, str) and pair_id:
            by_pair_id[pair_id] = row
    return by_pair_id


def get_source_input(output_row: dict[str, Any], instruction_lookup: dict[str, dict[str, Any]]) -> str:
    pair_id = output_row.get("pair_id", "")
    source_row = instruction_lookup.get(str(pair_id))
    if not source_row:
        raise KeyError(f"Could not find instruction input for pair_id={pair_id!r}")
    return str(source_row.get("input", ""))

In [4]:
def display_fitmyresume_output(output: dict[str, Any]) -> None:
    score = output.get("score", "N/A")
    explanation = output.get("explanation", {})
    suggestions = output.get("resume_suggestions", [])

    print("FIT SCORE")
    print(f"{score}/100")
    print()

    print("MATCHED QUALIFICATIONS")
    for item in explanation.get("matched_qualifications", []):
        print(f"- {item}")
    print()

    print("MISSING OR WEAK QUALIFICATIONS")
    for item in explanation.get("missing_or_weak_qualifications", []):
        print(f"- {item}")
    print()

    print("OVERALL REASONING")
    print(explanation.get("overall_reasoning", ""))
    print()

    print("RESUME SUGGESTIONS")
    for index, suggestion in enumerate(suggestions, start=1):
        if not isinstance(suggestion, dict):
            print(f"{index}. {suggestion}")
            continue
        section = suggestion.get("section", "section")
        action = suggestion.get("action", "revise")
        text = suggestion.get("suggestion", "")
        evidence = suggestion.get("evidence_from_resume", "")
        requirement = suggestion.get("job_requirement_addressed", "")
        print(f"{index}. [{section} / {action}] {text}")
        if evidence:
            print(f"   Evidence: {evidence}")
        if requirement:
            print(f"   Requirement addressed: {requirement}")

## Polished Presentation Example

This cell loads a representative parseable fine-tuned output, extracts the original resume and job description, checks input length, and displays the final user-facing sections.

## Custom Resume and Job Description

Run this cell to enter custom resume and job description text interactively. After loading the Transformers/PEFT model in the next cell, click **Generate model output** to score the custom input and display resume suggestions.


In [5]:
import ipywidgets as widgets
from IPython.display import clear_output, display

resume_box = widgets.Textarea(
    value="",
    placeholder="Paste your resume text here.",
    description="Resume",
    layout=widgets.Layout(width="100%", height="220px"),
)
job_box = widgets.Textarea(
    value="",
    placeholder="Paste the job description text here.",
    description="Job",
    layout=widgets.Layout(width="100%", height="220px"),
)
display(resume_box, job_box)


Textarea(value='', description='Resume', layout=Layout(height='220px', width='100%'), placeholder='Paste your …

Textarea(value='', description='Job', layout=Layout(height='220px', width='100%'), placeholder='Paste the job …

## Install Inference Dependencies

Run this in Colab if the runtime does not already have the Transformers/PEFT inference packages installed.


In [6]:
%pip install -q -U transformers peft accelerate bitsandbytes ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 159.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 150.2 MB/s eta 0:00:00


## Load Transformers/PEFT Model

Run this once before generating output for custom text. Use a GPU runtime. If your adapter path is different, update `ADAPTER_PATH`.


In [7]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = PROJECT_ROOT / "models/qwen25-7b-fitmyresume-lora-v2/final"
LOAD_IN_4BIT = True
TORCH_DTYPE = "bfloat16"
DEVICE_MAP = "auto"

FITMYRESUME_SYSTEM_PROMPT = (
    "Evaluate the resume against the job description. Return only valid JSON with "
    "score, explanation, and resume_suggestions. Do not invent experience."
)

def render_chat_prompt(tokenizer: Any, messages: list[dict[str, str]]) -> str:
    apply_chat_template = getattr(tokenizer, "apply_chat_template", None)
    if callable(apply_chat_template):
        try:
            return str(apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        except ValueError as error:
            if "chat_template" not in str(error):
                raise
    rendered_messages = []
    for message in messages:
        role = message.get("role", "user").upper()
        content = message.get("content", "")
        rendered_messages.append(f"{role}:\n{content}")
    return "\n\n".join(rendered_messages) + "\n\nASSISTANT:\n"

def resolve_torch_dtype(dtype: str) -> Any:
    if dtype == "auto":
        return "auto"
    import torch
    return getattr(torch, dtype)

def load_transformers_peft_model() -> tuple[Any, Any]:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_kwargs: dict[str, Any] = {
        "device_map": DEVICE_MAP,
        "torch_dtype": resolve_torch_dtype(TORCH_DTYPE),
    }
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=resolve_torch_dtype(TORCH_DTYPE),
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    tokenizer_source = ADAPTER_PATH if (ADAPTER_PATH / "tokenizer_config.json").exists() else BASE_MODEL
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, **model_kwargs)
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()
    return model, tokenizer

def parse_generated_json(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        if cleaned.startswith("json"):
            cleaned = cleaned[4:].strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        parsed = json.loads(cleaned[start : end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("Generated output JSON must be an object.")
    return parsed

model, tokenizer = load_transformers_peft_model()
print(f"Loaded Transformers/PEFT model with adapter: {ADAPTER_PATH}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Transformers/PEFT model with adapter: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora-v2/final


## Generate Custom Model Output

Click the button after entering custom text and loading the model. The generated JSON is parsed and displayed with the same FitMyResume output formatter used by the saved examples.


In [8]:
generate_button = widgets.Button(description="Generate model output", button_style="success")
generate_output = widgets.Output()

def generate_custom_output(_: widgets.Button) -> None:
    with generate_output:
        clear_output()
        custom_resume_text = resume_box.value.strip()
        custom_job_description = job_box.value.strip()
        validate_input_lengths(custom_resume_text, custom_job_description)
        if not custom_resume_text or not custom_job_description:
            raise ValueError("Paste both resume text and job description text before generating output.")

        messages = [
            {"role": "system", "content": FITMYRESUME_SYSTEM_PROMPT},
            {"role": "user", "content": f"RESUME:\n{custom_resume_text}\n\nJOB_DESCRIPTION:\n{custom_job_description}"},
        ]
        prompt = render_chat_prompt(tokenizer, messages)
        inputs = tokenizer(prompt, return_tensors="pt")
        if hasattr(inputs, "to"):
            model_device = getattr(model, "device", None)
            if model_device is None:
                model_device = next(model.parameters()).device
            inputs = inputs.to(model_device)

        prompt_token_count = len(inputs["input_ids"][0])
        pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        print("Generating model output...")
        import torch
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                pad_token_id=pad_token_id,
            )
        generated_text = tokenizer.decode(output_ids[0][prompt_token_count:], skip_special_tokens=True).strip()
        print("Raw model output:")
        print(generated_text)
        print("\nParsed FitMyResume output:")
        display_fitmyresume_output(parse_generated_json(generated_text))

generate_button.on_click(generate_custom_output)
display(generate_button, generate_output)


Button(button_style='success', description='Generate model output', style=ButtonStyle())

Output()

In [9]:
demo_rows = read_jsonl(RESULTS_FILE)
instruction_rows = read_jsonl(INSTRUCTIONS_FILE)
instruction_lookup = instruction_rows_by_pair_id(instruction_rows)
demo_row = find_demo_row(demo_rows)
demo_output = get_parsed_output(demo_row)
resume_text, job_description = split_instruction_input(get_source_input(demo_row, instruction_lookup))
validate_input_lengths(resume_text, job_description)

print(f"Pair ID: {demo_row.get('pair_id', '')}")
print(f"Backend: {demo_row.get('serving_backend', 'saved output')}")
print(f"Resume preview: {shorten(resume_text, width=350, placeholder=' ...')}")
print(f"Job preview: {shorten(job_description, width=350, placeholder=' ...')}")

Pair ID: validation_11773925_job_000274_strong_hybrid
Backend: transformers_peft
Resume preview: MARKETING & SALES ANALYST Summary A result oriented Marketing & Sales Analyst with 10 years of industry experience in the Staffing, Internet / ecommerce / Dotcom, Media / Publishing domain with expertise in Business process improvement, Business Process Flow Documentation & Data Analysis. Experienced with and proficient in Microsoft Office ...
Job preview: a client of insight global is seeking an enterprise sales executive this position is responsible for generating new sales through existing and selfgenerated leads and meeting or exceeding sales quotas the position is focused on the enterprise erp market including sage microsoft dynamics oracle sap ecc and others benefits offered medical dental ...


In [10]:
display_fitmyresume_output(demo_output)

FIT SCORE
65/100

MATCHED QUALIFICATIONS
- 10+ years of sales experience including inside and outside sales roles
- Proven track record of exceeding sales quotas (e.g., 'achieved targets for 15 quarters' and 'revenue growth of more than 40%')
- Experience with Salesforce for lead management and outreach
- Consultative sales approach demonstrated in multiple roles
- Bachelor's degree in International Business Management (Marketing)

MISSING OR WEAK QUALIFICATIONS
- No direct experience selling ERP software (Sage, Microsoft Dynamics, Oracle, SAP, ECC)
- Lack of enterprise software sales experience; most roles are in staffing/recruitment or media
- No mention of SaaS or managed services background
- No explicit experience calling on C-level executives or technology decision makers of large companies
- Weak evidence of maintaining a robust sales funnel or executing a structured sales process

OVERALL REASONING
The resume demonstrates strong general sales skills, quota attainment, and CRM p

## Try Another Saved Example

Change `example_index` to inspect a different saved fine-tuned output. The notebook filters to parseable rows first.

In [11]:
parseable_rows = [row for row in demo_rows if row.get("parse_success") and isinstance(row.get("parsed_output"), dict)]
example_index = 0

selected_row = parseable_rows[example_index]
selected_resume, selected_job = split_instruction_input(get_source_input(selected_row, instruction_lookup))
validate_input_lengths(selected_resume, selected_job)

print(f"Pair ID: {selected_row.get('pair_id', '')}")
print(f"Resume chars: {len(selected_resume):,}")
print(f"Job description chars: {len(selected_job):,}")
display_fitmyresume_output(get_parsed_output(selected_row))

Pair ID: validation_10149490_job_000176_strong_hybrid
Resume chars: 9,733
Job description chars: 3,282
FIT SCORE
35/100

MATCHED QUALIFICATIONS
- Managed construction projects including OSBL, ISBL, and FGS projects, demonstrating ability to oversee construction processes.
- Experience with budget control, cost forecasting, and resource allocation, aligning with financial oversight responsibilities.
- Supervised large crews (up to 500 employees) and managed multiple projects simultaneously, showing supervisory capability.
- Led safety programs and coordinated with operations, project engineering, and contractors, matching cross-functional coordination.

MISSING OR WEAK QUALIFICATIONS
- No experience in fast-food or restaurant construction, which is the domain of the job.
- Lack of a bachelor's degree; only high school diploma and some certifications.
- No mention of vendor invoice management, GC approval, or warranty management.
- No experience with real estate, design permitting, or IT

## Optional Live Inference

Use this command only in a GPU runtime with the base model and LoRA adapter available. It is not required for the saved-output notebook demo.

### Transformers/PEFT path

```powershell
python src/run_finetuned_transformers_inference.py `
  --input data/instruction_tuning/instruction_tuning_validation.jsonl `
  --output results/demo_transformers_outputs.jsonl `
  --adapter-path models/qwen25-7b-fitmyresume-lora-v2/final `
  --load-in-4bit `
  --limit 3
```

After generating a new output file, set `RESULTS_PATH` at the top of this notebook to the new JSONL path and rerun the display cells.